# PtyREX → Pelz probe & positions (batch)

Same conversion as `ptyrex_to_pelz.ipynb`, run over many scan folders. For every folder in `FOLDERS` it looks in
`<folder>/pty_out/initial_recon/`, converts **every** `.hdf` it finds, and writes one Pelz `.h5` per `.hdf` into `OUTPUT_FOLDER`:

| key | shape | meaning |
|---|---|---|
| `probe` | `(1, n_modes, M, M)` complex64 | real-space probe, centred in the window |
| `best_positions` | `(1, 2, K)` float32 | probe positions in Å from the field centre; row 0 = x, row 1 = y |

**Output name:** `<scan folder>_<hdf name>.h5`, e.g. `20260524_112119_ptycho_0_0_20260605-120024.h5`, so every file says which
folder it came from. A table of what was written (`conversion_summary.csv`) is saved next to them.

A folder with no `.hdf`, or a file that fails to read, is reported at the end and the batch carries on.

**Edit the settings cell, then run all cells.**

In [ ]:
# ---------------- Settings: edit these ----------------
FOLDERS = """
/dls/e02/data/2026/nt42695-2/processing/Merlin/CeO2_FSP_p2/20260524_112119
/dls/e02/data/2026/nt42695-2/processing/Merlin/CeO2_FSP_p2/20260524_112644
/dls/e02/data/2026/nt42695-2/processing/Merlin/CeO2_FSP_p2/20260524_113110
/dls/e02/data/2026/nt42695-2/processing/Merlin/CeO2_FSP_p2/20260524_113554
/dls/e02/data/2026/nt42695-2/processing/Merlin/CeO2_FSP_p2/20260524_114023
/dls/e02/data/2026/nt42695-2/processing/Merlin/CeO2_FSP_p2/20260524_114443
/dls/e02/data/2026/nt42695-2/processing/Merlin/CeO2_FSP_p2/20260524_114951
/dls/e02/data/2026/nt42695-2/processing/Merlin/CeO2_FSP_p2/20260524_115357
/dls/e02/data/2026/nt42695-2/processing/Merlin/CeO2_FSP_p2/20260524_120002
/dls/e02/data/2026/nt42695-2/processing/Merlin/CeO2_FSP_p2/20260524_120714
/dls/e02/data/2026/nt42695-2/processing/Merlin/CeO2_FSP_p2/20260524_121253
/dls/e02/data/2026/nt42695-2/processing/Merlin/CeO2_FSP_p2/20260524_123150
/dls/e02/data/2026/nt42695-2/processing/Merlin/CeO2_FSP_p2/20260524_123848
/dls/e02/data/2026/nt42695-2/processing/Merlin/CeO2_FSP_p2/20260524_124500
/dls/e02/data/2026/nt42695-2/processing/Merlin/CeO2_FSP_p2/20260524_125048
/dls/e02/data/2026/nt42695-2/processing/Merlin/CeO2_FSP_p2/20260524_125551
/dls/e02/data/2026/nt42695-2/processing/Merlin/CeO2_FSP_p2/20260524_130110
/dls/e02/data/2026/nt42695-2/processing/Merlin/CeO2_FSP_p2/20260524_130621
/dls/e02/data/2026/nt42695-2/processing/Merlin/CeO2_FSP_p2/20260524_131438
/dls/e02/data/2026/nt42695-2/processing/Merlin/CeO2_FSP_p2/20260524_132150
/dls/e02/data/2026/nt42695-2/processing/Merlin/CeO2_FSP_p2/20260524_132837
/dls/e02/data/2026/nt42695-2/processing/Merlin/CeO2_FSP_p2/20260524_142013
/dls/e02/data/2026/nt42695-2/processing/Merlin/CeO2_FSP_p2/20260524_142713
/dls/e02/data/2026/nt42695-2/processing/Merlin/CeO2_FSP_p2/20260524_143822
/dls/e02/data/2026/nt42695-2/processing/Merlin/CeO2_FSP_p2/20260524_144451
/dls/e02/data/2026/nt42695-2/processing/Merlin/CeO2_FSP_p2/20260524_144945
/dls/e02/data/2026/nt42695-2/processing/Merlin/CeO2_FSP_p2/20260524_145429
/dls/e02/data/2026/nt42695-2/processing/Merlin/CeO2_FSP_p2/20260524_145912
/dls/e02/data/2026/nt42695-2/processing/Merlin/CeO2_FSP_p2/20260524_150646
/dls/e02/data/2026/nt42695-2/processing/Merlin/CeO2_FSP_p2/20260524_151201
/dls/e02/data/2026/nt42695-2/processing/Merlin/CeO2_FSP_p2/20260524_152057
/dls/e02/data/2026/nt42695-2/processing/Merlin/CeO2_FSP_p2/20260524_152815
/dls/e02/data/2026/nt42695-2/processing/Merlin/CeO2_FSP_p2/20260524_153450
/dls/e02/data/2026/nt42695-2/processing/Merlin/CeO2_FSP_p2/20260524_154302
/dls/e02/data/2026/nt42695-2/processing/Merlin/CeO2_FSP_p2/20260524_154911
/dls/e02/data/2026/nt42695-2/processing/Merlin/CeO2_FSP_p2/20260524_155607
/dls/e02/data/2026/nt42695-2/processing/Merlin/CeO2_FSP_p2/20260524_160023
/dls/e02/data/2026/nt42695-2/processing/Merlin/CeO2_FSP_p2/20260524_160606
""".split()
# Or pick up every scan folder automatically instead of listing them:
# import glob; FOLDERS = sorted(glob.glob("/dls/e02/data/2026/nt42695-2/processing/Merlin/CeO2_FSP_p2/20260524_*"))

RECON_SUBFOLDER = "pty_out/initial_recon"   # where the PtyREX .hdf files sit inside each folder
OUTPUT_FOLDER   = "/dls/e02/data/2026/nt42695-2/processing/Merlin/End_to_End_CeO2_FSP_P2/Alignment_Results"   # the Pelz notebook's probe_position_path

WINDOW   = None   # probe window in pixels. None = keep PtyREX size.
                  # Must equal the detector size of your zarr patterns (e.g. 150 if you binned 450 by 3).
N_MODES  = None   # keep only the first N probe modes (None = all). Every tilt must end up with the same number.
USE_REFINED_POSITIONS = True   # False = nominal scan grid instead of PtyREX's refined positions
OVERWRITE = True  # False = skip outputs that already exist (useful if a run stopped part-way)

In [ ]:
import os
import csv
import glob
import traceback
import numpy as np
import h5py
import matplotlib.pyplot as plt

os.makedirs(OUTPUT_FOLDER, exist_ok=True)
MARGIN = 5   # same margin as the Pelz notebook


def output_name(folder, ptyrex_file):
    """<scan folder>_<hdf name>.h5, e.g. 20260524_112119_ptycho_0_0_20260605-120024.h5"""
    folder_name = os.path.basename(os.path.normpath(folder))
    stem = os.path.splitext(os.path.basename(ptyrex_file))[0]
    return f"{folder_name}_{stem}.h5"


def convert(ptyrex_file, output_file, folder):
    """Convert one PtyREX .hdf to a Pelz .h5. Returns a dict of numbers for the summary."""
    # ---- 1. read (only slice 0 / copy 0 of the probe: the entrance-plane, final probe) ----
    with h5py.File(ptyrex_file, "r") as f:
        out = f["entry_1/process_1/output_1"]
        probe       = out["probe"][:, 0, 0, 0, 0]                  # (modes, N, N)
        object_size = out["object"].shape[-2:]
        pos_refined = out["scan_positions"][()][0].astype(np.float64)
        pos_nominal = f["entry_1/experiment_1/scan/positions"][()][0].astype(np.float64)
        dx_A        = float(np.ravel(f["entry_1/process_1/common_1/dx"][()])[0]) * 1e10      # m -> Angstrom
        energy_eV   = float(np.ravel(f["entry_1/experiment_1/source/energy"][()])[0])

    # ---- 2. probe: choose modes, crop the window ----
    if N_MODES is not None:
        probe = probe[:N_MODES]
    N = probe.shape[-1]
    M = N if WINDOW is None else WINDOW
    if M > N:
        raise ValueError(f"WINDOW = {M} px is larger than the PtyREX probe ({N} px)")
    start = N // 2 - M // 2                                         # keep the probe centred at M/2
    probe_out = probe[:, start:start + M, start:start + M].astype(np.complex64)
    power = (np.abs(probe) ** 2).sum((1, 2))

    # ---- 3. positions: pixels -> Angstrom from the field centre, x first ----
    pos = pos_refined if USE_REFINED_POSITIONS else pos_nominal
    centre_y, centre_x = (np.array(object_size) - 1) / 2
    x_A = (pos[1] - centre_x) * dx_A
    y_A = (pos[0] - centre_y) * dx_A
    best_positions = np.stack([x_A, y_A]).astype(np.float32)[None]    # (1, 2, K)
    shift = np.hypot(*(pos_refined - pos_nominal)) * dx_A

    # ---- 4. save ----
    with h5py.File(output_file, "w") as f:
        f["probe"] = probe_out[None]                 # (1, modes, M, M)
        f["best_positions"] = best_positions         # (1, 2, K)
        f.attrs["source"] = os.path.abspath(ptyrex_file)
        f.attrs["source_folder"] = os.path.abspath(folder)
        f.attrs["dx_A"] = dx_A
        f.attrs["energy_eV"] = energy_eV
        f.attrs["refined_positions"] = USE_REFINED_POSITIONS
        f.attrs["centre_obj_px"] = (centre_y, centre_x)

    # ---- 5. check it reads back the way the Pelz notebook reads it ----
    with h5py.File(output_file, "r") as f:
        assert f["probe"].shape == (1, probe_out.shape[0], M, M)
        assert f["best_positions"].shape == best_positions.shape

    # smallest scan_shape (even) that keeps every probe window inside the volume
    need = 2 * int(np.ceil(max(np.abs(best_positions[0]).max() / dx_A - MARGIN + 0.5, 0)))

    return {
        "folder": os.path.basename(os.path.normpath(folder)),
        "ptyrex_file": os.path.abspath(ptyrex_file),
        "output_file": os.path.abspath(output_file),
        "modes": probe_out.shape[0],
        "window_px": M,
        "K": best_positions.shape[-1],
        "dx_A": dx_A,
        "energy_eV": energy_eV,
        "scan_shape_needed": need,
        "mean_shift_A": shift.mean(),
        "max_shift_A": shift.max(),
        "mode_power_pct": " ".join(f"{p:.1f}" for p in 100 * power / power.sum()),
        "_intensity": (np.abs(probe_out) ** 2).sum(0),          # for the preview plot only
    }

## 1. Find the PtyREX files

In [ ]:
FOLDERS = list(dict.fromkeys(os.path.normpath(p) for p in FOLDERS))   # drop accidental duplicates, keep order

jobs = []            # (folder, ptyrex .hdf, output .h5)
no_files = []        # folders with nothing to convert
for folder in FOLDERS:
    recon_dir = os.path.join(folder, RECON_SUBFOLDER)
    hdfs = sorted(glob.glob(os.path.join(recon_dir, "*.hdf")))
    if not os.path.isdir(recon_dir):
        no_files.append((folder, f"no {RECON_SUBFOLDER} folder"))
    elif not hdfs:
        no_files.append((folder, f"no .hdf in {RECON_SUBFOLDER}"))
    for h in hdfs:
        jobs.append((folder, h, os.path.join(OUTPUT_FOLDER, output_name(folder, h))))

print(f"{len(FOLDERS)} folders, {len(jobs)} .hdf files to convert")
for folder in FOLDERS:
    n = sum(j[0] == folder for j in jobs)
    print(f"  {os.path.basename(folder)}: {n} file{'s' if n != 1 else ''}")
for folder, why in no_files:
    print(f"  WARNING {os.path.basename(folder)}: {why}")

## 2. Convert

In [ ]:
results, failed, skipped = [], [], []
for i, (folder, ptyrex_file, output_file) in enumerate(jobs, 1):
    tag = f"[{i}/{len(jobs)}] {os.path.basename(output_file)}"
    if not OVERWRITE and os.path.exists(output_file):
        skipped.append(output_file)
        print(f"{tag}: exists, skipped")
        continue
    try:
        r = convert(ptyrex_file, output_file, folder)
        results.append(r)
        print(f"{tag}: probe ({r['modes']}, {r['window_px']}, {r['window_px']}), K = {r['K']}, "
              f"dx = {r['dx_A']:.5f} A, refined shift max {r['max_shift_A']:.2f} A")
    except Exception as e:
        failed.append((ptyrex_file, f"{type(e).__name__}: {e}"))
        print(f"{tag}: FAILED  {type(e).__name__}: {e}")

## 3. Summary

Writes `conversion_summary.csv` (folder → PtyREX file → output file, plus the numbers) and checks that every file
agrees on the things the Pelz notebook needs to be the same for all tilts.

In [ ]:
summary_csv = os.path.join(OUTPUT_FOLDER, "conversion_summary.csv")
columns = [k for k in (results[0] if results else {}) if not k.startswith("_")]
if results:
    with open(summary_csv, "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=columns, extrasaction="ignore")
        w.writeheader()
        for r in results:
            w.writerow({k: (f"{v:.6g}" if isinstance(v, float) else v) for k, v in r.items() if k in columns})
    print("wrote", summary_csv)

print(f"\nconverted {len(results)}, skipped {len(skipped)}, failed {len(failed)}, folders with no .hdf {len(no_files)}")
for path, err in failed:
    print("  FAILED ", path, "\n         ", err)
for folder, why in no_files:
    print("  EMPTY  ", folder, f"({why})")

if results:
    def check(key, name, fix=""):
        vals = sorted({r[key] for r in results})
        if len(vals) == 1:
            print(f"{name:14s} all {vals[0]}")
        else:
            print(f"{name:14s} WARNING differs between files: {vals} {fix}")
            for v in vals:
                print(f"{'':16s}{v}: " + ", ".join(r["folder"] for r in results if r[key] == v))
    print()
    check("modes", "probe modes", "-> set N_MODES so every tilt has the same number")
    check("window_px", "window (M1)", "-> set WINDOW")
    check("energy_eV", "energy (eV)")
    dxs = np.array([r["dx_A"] for r in results])
    print(f"{'dx (A)':14s} {dxs.min():.5f} to {dxs.max():.5f}" +
          ("" if dxs.max() / dxs.min() - 1 < 0.01 else "   WARNING: differs by more than 1%"))

    need = max(r["scan_shape_needed"] for r in results)
    print(f"""
Settings for the Pelz notebook (largest scan_shape over all files):
    M1 = {results[0]['window_px']}
    scan_shape = [{need}, {need}]
    E = {results[0]['energy_eV']:.0f}
    select_inds = {{t: 0 for t in take}}
""")

## 4. Quick look at every probe

Total intensity (all modes summed) of each converted probe, titled with its scan folder. A failed reconstruction usually
stands out here.

In [ ]:
if results:
    ncol = min(8, len(results))
    nrow = int(np.ceil(len(results) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(2.1 * ncol, 2.3 * nrow), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, r in zip(axes.flat, results):
        ax.imshow(r["_intensity"], cmap="magma")
        stem = os.path.splitext(os.path.basename(r["ptyrex_file"]))[0]
        ax.set_title(f"{r['folder']}\n{stem}", fontsize=6)
    plt.tight_layout(); plt.show()